# 10 — H3 Horizon-Specific Hybrid

## Purpose
Script 09 reached strong development performance, but the remaining near-ties are concentrated at the 3-day horizon. This notebook therefore applies **one common H3 methodology to USD, GBP, EUR and JPY**.

It does not hand-tune individual losing currencies.

### Design
For each outer year:
- previous year = inner validation;
- training targets must end before inner-validation start;
- same model/feature/window/shrinkage/blend search for all four currencies;
- dedicated H3 model predicts 3-day change;
- final H3 forecast can combine Random Walk, Script 09 and the dedicated H3 model;
- 2024–2025 remains untouched.

After this notebook, freeze the H3 development design.


In [ ]:

from pathlib import Path
import json, warnings, time
import numpy as np
import pandas as pd
from scipy.stats import t as student_t

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, HuberRegressor, ElasticNet
from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 230)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

CURRENCIES=["USD","GBP","EUR","JPY"]
H=3
YEARS=[2019,2020,2021,2022,2023]
FEATURE_FAMILIES=["TT_OWN","TT_CROSS","FULL_PIT"]
TRAIN_WINDOWS=["RECENT_2Y","RECENT_3Y","RECENT_5Y","EXPANDING"]
MODEL_KEYS=["RIDGE_1","RIDGE_10","HUBER","ELASTIC_NET","EXTRA_TREES","HIST_GB"]
SHRINK=[0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.75,1.0]
BLEND=[0.0,0.25,0.5,0.75,1.0]
RANDOM_STATE=42


In [ ]:

def find_root():
    for root in [Path.cwd(),Path.cwd().parent,Path.cwd().parent.parent]:
        dev=root/"data"/"processed"/"02_development_model_dataset.csv"
        feat=root/"data"/"processed"/"02_feature_sets.json"
        rw=root/"outputs"/"predictions"/"04_random_walk_predictions_2019_2023.csv"
        stat=root/"outputs"/"predictions"/"05_statistical_predictions_2019_2023.csv"
        if dev.exists() and feat.exists() and rw.exists() and stat.exists():
            return root.resolve()
    raise FileNotFoundError("Required Script 02/04/05 outputs not found.")

ROOT=find_root()
DEV_PATH=ROOT/"data"/"processed"/"02_development_model_dataset.csv"
FEATURE_PATH=ROOT/"data"/"processed"/"02_feature_sets.json"
PRED_DIR=ROOT/"outputs"/"predictions"
TABLE_DIR=ROOT/"outputs"/"tables"
TABLE_DIR.mkdir(parents=True,exist_ok=True)

RW_PATH=PRED_DIR/"04_random_walk_predictions_2019_2023.csv"
STAT_PATH=PRED_DIR/"05_statistical_predictions_2019_2023.csv"
ML_PATH=next((p for p in [PRED_DIR/"06_FAST_ml_predictions_2019_2023.csv",
                          PRED_DIR/"06_ml_predictions_2019_2023.csv"] if p.exists()),None)
ONLINE09_PATH=PRED_DIR/"09_online_hybrid_predictions_2019_2023.csv"

if ML_PATH is None: raise FileNotFoundError("Script 06 prediction file not found.")
if not ONLINE09_PATH.exists(): raise FileNotFoundError("Script 09 prediction file not found.")

df=pd.read_csv(DEV_PATH,parse_dates=["Date"])
df["Target_Date_H3"]=pd.to_datetime(df["Target_Date_H3"],errors="coerce")
with open(FEATURE_PATH,"r",encoding="utf-8") as f:
    FEATURE_SETS=json.load(f)

assert df["Date"].max()<pd.Timestamp("2024-01-01")
print("PASS — 2024–2025 holdout not loaded.")


In [ ]:

rw=pd.read_csv(RW_PATH,parse_dates=["Forecast_Origin","Target_Date"])
stat=pd.read_csv(STAT_PATH,parse_dates=["Forecast_Origin","Target_Date"])
ml=pd.read_csv(ML_PATH,parse_dates=["Forecast_Origin","Target_Date"])
on9=pd.read_csv(ONLINE09_PATH,parse_dates=["Forecast_Origin","Target_Date"])

def pick(frame,cands):
    for c in cands:
        if c in frame.columns: return c
    raise ValueError(f"Missing prediction column. Found: {list(frame.columns)}")

stat_col=pick(stat,["Stat_Prediction","Statistical_Prediction","Model_Prediction","Prediction"])
ml_col=pick(ml,["ML_Prediction","Model_Prediction","Prediction"])
on9_col=pick(on9,["Online_Hybrid_Prediction","Ensemble_Prediction","Prediction"])

KEYS=["Currency","Horizon","Forecast_Origin"]
experts=rw[KEYS+["Target_Date","Actual","RW_Prediction"]].copy()
experts=experts[experts["Horizon"]==3].copy()
experts=experts.merge(stat[KEYS+[stat_col]].rename(columns={stat_col:"STAT"}),on=KEYS,how="inner")
experts=experts.merge(ml[KEYS+[ml_col]].rename(columns={ml_col:"ML"}),on=KEYS,how="inner")
experts=experts.merge(on9[KEYS+[on9_col]].rename(columns={on9_col:"ONLINE09"}),on=KEYS,how="inner")
experts["Validation_Year"]=experts["Forecast_Origin"].dt.year
print("Matched H3 expert rows:",len(experts))


In [ ]:

def rmse(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    m=np.isfinite(y)&np.isfinite(p)
    return float(np.sqrt(np.mean((y[m]-p[m])**2))) if m.sum() else np.nan

def mae(y,p):
    y=np.asarray(y,float); p=np.asarray(p,float)
    m=np.isfinite(y)&np.isfinite(p)
    return float(np.mean(np.abs(y[m]-p[m]))) if m.sum() else np.nan

def skill(m,r): return float(1-m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan
def theils_u(m,r): return float(m/r) if np.isfinite(m) and np.isfinite(r) and r!=0 else np.nan

def dm_test(y,model,rw_pred,horizon=3):
    y=np.asarray(y,float); model=np.asarray(model,float); rw_pred=np.asarray(rw_pred,float)
    m=np.isfinite(y)&np.isfinite(model)&np.isfinite(rw_pred)
    y=y[m]; model=model[m]; rw_pred=rw_pred[m]
    n=len(y)
    if n<20: return np.nan,np.nan
    d=(y-rw_pred)**2-(y-model)**2
    lag=max(0,horizon-1)
    lrv=np.var(d,ddof=1)
    for k in range(1,lag+1):
        if len(d)<=k: break
        cov=np.cov(d[k:],d[:-k],ddof=1)[0,1]
        lrv += 2*(1-k/(lag+1))*cov
    if not np.isfinite(lrv) or lrv<=0: return np.nan,np.nan
    stat=np.mean(d)/np.sqrt(lrv/n)
    return float(stat),float(2*student_t.sf(abs(stat),df=n-1))


## H3-specific features
All derived features use only current/past information.

In [ ]:

for cur in CURRENCIES:
    mid=df[f"{cur}_mid"]
    r=np.log(mid/mid.shift(1))
    df[f"{cur}_h3_ret1"]=r
    for w in [2,3,5,7]:
        df[f"{cur}_h3_ret{w}sum"]=r.rolling(w).sum()
    for w in [3,5,10]:
        df[f"{cur}_h3_vol{w}"]=r.rolling(w).std()
        ma=mid.rolling(w).mean()
        df[f"{cur}_h3_ma{w}gap"]=(mid-ma)/mid

def features_for(cur,family):
    feats=[c for c in FEATURE_SETS[cur][family] if c in df.columns]
    own=[f"{cur}_h3_ret1",f"{cur}_h3_ret2sum",f"{cur}_h3_ret3sum",
         f"{cur}_h3_ret5sum",f"{cur}_h3_ret7sum",
         f"{cur}_h3_vol3",f"{cur}_h3_vol5",f"{cur}_h3_vol10",
         f"{cur}_h3_ma3gap",f"{cur}_h3_ma5gap",f"{cur}_h3_ma10gap"]
    cross=[]
    for other in CURRENCIES:
        if other!=cur:
            cross += [f"{other}_h3_ret1",f"{other}_h3_ret3sum",f"{other}_h3_vol5"]
    return list(dict.fromkeys([c for c in feats+own+cross if c in df.columns]))


In [ ]:

def model(key):
    if key=="RIDGE_1":
        return Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler()),("m",Ridge(alpha=1.0))])
    if key=="RIDGE_10":
        return Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler()),("m",Ridge(alpha=10.0))])
    if key=="HUBER":
        return Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler()),
                         ("m",HuberRegressor(epsilon=1.35,alpha=0.001,max_iter=500))])
    if key=="ELASTIC_NET":
        return Pipeline([("i",SimpleImputer(strategy="median")),("s",StandardScaler()),
                         ("m",ElasticNet(alpha=0.001,l1_ratio=0.2,max_iter=5000,random_state=42))])
    if key=="EXTRA_TREES":
        return Pipeline([("i",SimpleImputer(strategy="median")),
                         ("m",ExtraTreesRegressor(n_estimators=140,max_depth=6,min_samples_leaf=10,
                                                  max_features=.7,random_state=42,n_jobs=-1))])
    if key=="HIST_GB":
        return Pipeline([("i",SimpleImputer(strategy="median")),
                         ("m",HistGradientBoostingRegressor(learning_rate=.03,max_iter=150,
                                                            max_leaf_nodes=15,l2_regularization=3.0,
                                                            random_state=42))])
    raise ValueError(key)

def apply_window(train,start,window):
    years={"RECENT_2Y":2,"RECENT_3Y":3,"RECENT_5Y":5}
    if window in years:
        return train[train["Date"]>=start-pd.DateOffset(years=years[window])].copy()
    return train.copy()

def batch(cur,family,window,val_year):
    feats=features_for(cur,family)
    target=f"{cur}_change_H3"
    start=pd.Timestamp(f"{val_year}-01-01")
    end=pd.Timestamp(f"{val_year+1}-01-01")
    train=df[(df["Date"]<start)&df[target].notna()&
             df["Target_Date_H3"].notna()&(df["Target_Date_H3"]<start)].copy()
    train=apply_window(train,start,window)
    val=df[(df["Date"]>=start)&(df["Date"]<end)&
           df[target].notna()&df["Target_Date_H3"].notna()].copy()
    return train,val,feats,target


In [ ]:

def tune_dedicated(cur,outer_year):
    inner=outer_year-1
    rows=[]
    for fam in FEATURE_FAMILIES:
        for win in TRAIN_WINDOWS:
            tr,va,feats,target=batch(cur,fam,win,inner)
            if len(tr)<200 or len(va)<30 or not feats: continue
            actual=va[f"{cur}_target_H3"].to_numpy(float)
            rwv=va[f"{cur}_mid"].to_numpy(float)
            for mk in MODEL_KEYS:
                try:
                    m=model(mk); m.fit(tr[feats],tr[target])
                    raw=rwv+m.predict(va[feats])
                except Exception:
                    continue
                for lam in SHRINK:
                    pred=rwv+lam*(raw-rwv)
                    rows.append({"Feature_Family":fam,"Train_Window":win,"Model":mk,
                                 "Lambda":lam,"Inner_RMSE":rmse(actual,pred)})
    tab=pd.DataFrame(rows)
    if tab.empty: return None,tab
    b=tab.sort_values("Inner_RMSE").iloc[0]
    return {"Feature_Family":b["Feature_Family"],"Train_Window":b["Train_Window"],
            "Model":b["Model"],"Lambda":float(b["Lambda"]),
            "Inner_RMSE":float(b["Inner_RMSE"])},tab

def predict_outer(cur,year,sel):
    tr,va,feats,target=batch(cur,sel["Feature_Family"],sel["Train_Window"],year)
    e=experts[(experts["Currency"]==cur)&(experts["Validation_Year"]==year)][
        ["Forecast_Origin","Target_Date","Actual","RW_Prediction","STAT","ML","ONLINE09"]].copy()
    va=va.merge(e,left_on="Date",right_on="Forecast_Origin",how="inner")
    m=model(sel["Model"]); m.fit(tr[feats],tr[target])
    change=m.predict(va[feats])
    rwv=va["RW_Prediction"].to_numpy(float)
    dedicated=rwv+sel["Lambda"]*change
    return pd.DataFrame({
        "Currency":cur,"Horizon":3,"Validation_Year":year,
        "Forecast_Origin":va["Forecast_Origin"],"Target_Date":va["Target_Date"],
        "Actual":va["Actual"],"RW_Prediction":va["RW_Prediction"],
        "STAT":va["STAT"],"ML":va["ML"],"ONLINE09":va["ONLINE09"],
        "DEDICATED_H3":dedicated
    })


## Blend rule
The same blend candidates are available to all four currencies.

In [ ]:

def choose_blend(cur,year,prior_dedicated):
    inner=year-1
    e=experts[(experts["Currency"]==cur)&(experts["Validation_Year"]==inner)][
        ["Forecast_Origin","Actual","RW_Prediction","STAT","ML","ONLINE09"]].copy()
    d=prior_dedicated[prior_dedicated["Validation_Year"]==inner][
        ["Forecast_Origin","DEDICATED_H3"]].copy()
    x=e.merge(d,on="Forecast_Origin",how="inner")
    if x.empty:
        return {"Blend_Type":"ONLINE09_ONLY","Alpha":1.0,"Inner_RMSE":np.nan},pd.DataFrame()

    y=x["Actual"].to_numpy(float)
    rows=[]
    singles={"RW_ONLY":"RW_Prediction","ONLINE09_ONLY":"ONLINE09","DEDICATED_ONLY":"DEDICATED_H3"}
    for name,col in singles.items():
        rows.append({"Blend_Type":name,"Alpha":1.0,"Inner_RMSE":rmse(y,x[col])})

    pairs={"RW_DEDICATED":("RW_Prediction","DEDICATED_H3"),
           "ONLINE09_DEDICATED":("ONLINE09","DEDICATED_H3"),
           "STAT_DEDICATED":("STAT","DEDICATED_H3"),
           "ML_DEDICATED":("ML","DEDICATED_H3")}

    for name,(a,b) in pairs.items():
        pa=x[a].to_numpy(float); pb=x[b].to_numpy(float)
        for alpha in BLEND:
            rows.append({"Blend_Type":name,"Alpha":alpha,
                         "Inner_RMSE":rmse(y,(1-alpha)*pa+alpha*pb)})

    tab=pd.DataFrame(rows)
    b=tab.sort_values("Inner_RMSE").iloc[0]
    return {"Blend_Type":b["Blend_Type"],"Alpha":float(b["Alpha"]),
            "Inner_RMSE":float(b["Inner_RMSE"])},tab

def apply_blend(x,btype,alpha):
    if btype=="RW_ONLY": return x["RW_Prediction"].to_numpy(float)
    if btype=="ONLINE09_ONLY": return x["ONLINE09"].to_numpy(float)
    if btype=="DEDICATED_ONLY": return x["DEDICATED_H3"].to_numpy(float)
    pairs={"RW_DEDICATED":("RW_Prediction","DEDICATED_H3"),
           "ONLINE09_DEDICATED":("ONLINE09","DEDICATED_H3"),
           "STAT_DEDICATED":("STAT","DEDICATED_H3"),
           "ML_DEDICATED":("ML","DEDICATED_H3")}
    a,b=pairs[btype]
    return (1-alpha)*x[a].to_numpy(float)+alpha*x[b].to_numpy(float)


In [ ]:

all_out=[]; model_sel=[]; blend_sel=[]; model_tabs=[]; blend_tabs=[]
dedicated_history={c:[] for c in CURRENCIES}

for cur in CURRENCIES:
    print(f"\\n===== {cur} H3 =====",flush=True)
    for year in YEARS:
        t0=time.time()
        sel,mtab=tune_dedicated(cur,year)
        if sel is None:
            sel={"Feature_Family":"TT_OWN","Train_Window":"EXPANDING",
                 "Model":"RIDGE_10","Lambda":0.0,"Inner_RMSE":np.nan}

        out_year=predict_outer(cur,year,sel)
        dedicated_history[cur].append(out_year)

        if not mtab.empty:
            z=mtab.copy(); z["Currency"]=cur; z["Outer_Year"]=year; model_tabs.append(z)
        model_sel.append({"Currency":cur,"Outer_Year":year,**sel})

        if year==YEARS[0]:
            bsel={"Blend_Type":"ONLINE09_ONLY","Alpha":1.0,"Inner_RMSE":np.nan}
            btab=pd.DataFrame()
        else:
            prior=pd.concat(dedicated_history[cur][:-1],ignore_index=True)
            bsel,btab=choose_blend(cur,year,prior)

        out_year["Final_H3_Prediction"]=apply_blend(out_year,bsel["Blend_Type"],bsel["Alpha"])
        out_year["Selected_Blend"]=bsel["Blend_Type"]
        out_year["Blend_Alpha"]=bsel["Alpha"]

        all_out.append(out_year)
        blend_sel.append({"Currency":cur,"Outer_Year":year,**bsel})

        if not btab.empty:
            z=btab.copy(); z["Currency"]=cur; z["Outer_Year"]=year; blend_tabs.append(z)

        print(f"{year} | {sel['Model']} | {sel['Feature_Family']} | {sel['Train_Window']} "
              f"| λ={sel['Lambda']:.2f} | {bsel['Blend_Type']} α={bsel['Alpha']:.2f} "
              f"| {time.time()-t0:.1f}s",flush=True)

final_predictions=pd.concat(all_out,ignore_index=True)
model_selections=pd.DataFrame(model_sel)
blend_selections=pd.DataFrame(blend_sel)


In [ ]:

yearly=[]
for (cur,year),g in final_predictions.groupby(["Currency","Validation_Year"]):
    mr=rmse(g["Actual"],g["Final_H3_Prediction"])
    ma=mae(g["Actual"],g["Final_H3_Prediction"])
    rr=rmse(g["Actual"],g["RW_Prediction"])
    _,p=dm_test(g["Actual"],g["Final_H3_Prediction"],g["RW_Prediction"],3)
    yearly.append({"Currency":cur,"Horizon":3,"Validation_Year":year,"Observations":len(g),
                   "RMSE":mr,"MAE":ma,"Forecast_Skill_vs_RW":skill(mr,rr),
                   "Theils_U":theils_u(mr,rr),"DM_pvalue_vs_RW":p})

yearly_results=pd.DataFrame(yearly).sort_values(["Currency","Validation_Year"])
display(yearly_results)
yearly_results.to_csv(TABLE_DIR/"10_H3_yearly_results.csv",index=False)


In [ ]:

pooled=[]
for cur,g in final_predictions.groupby("Currency"):
    mr=rmse(g["Actual"],g["Final_H3_Prediction"])
    ma=mae(g["Actual"],g["Final_H3_Prediction"])
    rr=rmse(g["Actual"],g["RW_Prediction"])
    _,p=dm_test(g["Actual"],g["Final_H3_Prediction"],g["RW_Prediction"],3)
    pooled.append({"Currency":cur,"Horizon":3,"Observations":len(g),
                   "RMSE":mr,"MAE":ma,"Forecast_Skill_vs_RW":skill(mr,rr),
                   "Theils_U":theils_u(mr,rr),"DM_pvalue_vs_RW":p})

pooled_results=pd.DataFrame(pooled).sort_values("Currency")
pooled_results["Beats_RW"]=pooled_results["Forecast_Skill_vs_RW"]>0
display(pooled_results)
print("H3 tasks beating RW:",int(pooled_results["Beats_RW"].sum()),"/",len(pooled_results))
pooled_results.to_csv(TABLE_DIR/"10_H3_pooled_results.csv",index=False)


In [ ]:

comparison=[]
for cur in CURRENCIES:
    g10=final_predictions[final_predictions["Currency"]==cur]
    g09=experts[experts["Currency"]==cur]
    rr=rmse(g10["Actual"],g10["RW_Prediction"])
    r09=rmse(g09["Actual"],g09["ONLINE09"])
    r10=rmse(g10["Actual"],g10["Final_H3_Prediction"])
    comparison.append({"Currency":cur,"RW_RMSE":rr,
                       "Script09_H3_RMSE":r09,"Script10_H3_RMSE":r10,
                       "Script09_Skill":skill(r09,rr),"Script10_Skill":skill(r10,rr),
                       "Script10_vs_09_RMSE_Improvement":1-r10/r09})
comparison=pd.DataFrame(comparison)
display(comparison)
comparison.to_csv(TABLE_DIR/"10_H3_vs_script09_comparison.csv",index=False)


In [ ]:

final_predictions.to_csv(PRED_DIR/"10_H3_predictions_2019_2023.csv",index=False)
model_selections.to_csv(TABLE_DIR/"10_H3_model_selections.csv",index=False)
blend_selections.to_csv(TABLE_DIR/"10_H3_blend_selections.csv",index=False)

if model_tabs:
    pd.concat(model_tabs,ignore_index=True).to_csv(TABLE_DIR/"10_H3_model_candidates.csv",index=False)
if blend_tabs:
    pd.concat(blend_tabs,ignore_index=True).to_csv(TABLE_DIR/"10_H3_blend_candidates.csv",index=False)

quality_gate=pd.DataFrame([
    {"Area":"Scope","Status":"PASS","Evidence":"One H3 methodology applied to all 4 currencies"},
    {"Area":"Holdout isolation","Status":"PASS","Evidence":"2024–2025 not loaded"},
    {"Area":"Target overlap","Status":"PASS","Evidence":"Training target dates precede validation-year start"},
    {"Area":"Inner model selection","Status":"PASS","Evidence":"Previous-year validation only"},
    {"Area":"Common search space","Status":"PASS","Evidence":"Same models/features/windows/shrinkage/blends for every currency"},
    {"Area":"Frozen RW","Status":"PASS","Evidence":"Script 04 Random Walk reused"},
])
display(quality_gate)
quality_gate.to_csv(TABLE_DIR/"10_H3_quality_gate.csv",index=False)

print("\\nIMPORTANT: freeze H3 model-development design after Script 10.")
print("2024–2025 remains untouched.")
